In [1]:
import os
import csv
import time
import psutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from itertools import combinations

import optuna
import logging
import sys

from functools import partial


# --- Reusable Functions ---
from ReusableFunctions.DataPreprocessing import DataPreprocessing
from ReusableFunctions.EvaluationMetrics import EvaluationMetrics as EM
from reproducibility_settings import set_global_seed
from ReusableFunctions.RecordBestModel import record_best_models



# Reproducibility

set_global_seed( seed=42, framework='torch' )


# Device

device = torch.device( "cuda" if torch.cuda.is_available() else "cpu" )

if torch.cuda.is_available():
    print( "GPU Name:", torch.cuda.get_device_name(0) )

else:
    print( "Running on CPU" )


# Benchmarking Helper Functions


def synchronize_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# ------------------------------------------------------------
# Parameter Count
# ------------------------------------------------------------

def get_parameter_counts(model):
    total_parameters = sum( p.numel() for p in model.parameters() )

    trainable_parameters = sum( p.numel() for p in model.parameters() if p.requires_grad )

    return ( total_parameters, trainable_parameters )

# ------------------------------------------------------------
# Linear Regression MACs
# ------------------------------------------------------------

def estimate_linear_macs_per_sample(model):    
    input_features = ( model.linear.in_features )

    output_features = ( model.linear.out_features )

    macs = ( input_features * output_features )

    return int(macs)

# ------------------------------------------------------------
# Inference Benchmark
# ------------------------------------------------------------

def measure_inference_time( model, X, batch_size=256, warmup=True ):
    if X is None:
        return ( np.nan, np.nan )

    if len(X) == 0:
        return ( np.nan, np.nan )

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------
    dataset = TensorDataset( torch.tensor( X, dtype=torch.float32 ) )

    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------
    loader = DataLoader( dataset, batch_size=batch_size, shuffle=False )

    model.eval()

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------
    if warmup:
        first_batch = ( next(iter(loader))[0].to(device) )

        with torch.no_grad():
            for _ in range(3):
                _ = model( first_batch )

        synchronize_device()

    # --------------------------------------------------------
    # Actual Timing
    # --------------------------------------------------------
    synchronize_device()

    inference_start = ( time.perf_counter() )

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to( device )

            _ = model( xb )

    synchronize_device()

    total_inference_time = ( time.perf_counter() - inference_start )

    latency_per_sample = ( total_inference_time / len(X) )

    return ( total_inference_time, latency_per_sample )



# Linear Regression Model


class LinearRegressionModel(nn.Module):
    def __init__( self, input_size ):

        super().__init__()

        self.linear = nn.Linear( input_size, 1 )

    def forward( self, x ):
        x = x.reshape( x.size(0), -1 )
        return self.linear( x )


# Train & Evaluate


def train_and_evaluate_model(
    model,
    train_loader,
    val_data,
    optimizer,
    criterion,
    scaler_y,
    forecast_window,
    epochs,
    patience=0.2
):

    # --------------------------------------------------------
    # Early stopping variables
    # --------------------------------------------------------
    best_val_loss = ( float('inf') )

    epochs_no_improve = 0

    actual_patience = (
        int( patience * epochs )

        if isinstance( patience, float )

        else patience
    )

    # Prevent patience = 0
    actual_patience = max( 1, actual_patience )

    # --------------------------------------------------------
    # Validation tensors
    # --------------------------------------------------------
    X_val_tensor = torch.tensor( val_data[0], dtype=torch.float32 ).to(device)

    y_val_tensor = torch.tensor( val_data[1], dtype=torch.float32 ).to(device)

    
    # Training Timer
    
    synchronize_device()

    training_start = ( time.perf_counter() )

    epochs_completed = 0

    
    # Epoch Loop
    
    for epoch in range( epochs ):

        # 
        # Training
        # 
        model.train()

        for xb, yb in train_loader:

            # ------------------------------------------------
            # CPU -> GPU
            # ------------------------------------------------
            xb = xb.to( device )

            yb = yb.to( device )                

            # ------------------------------------------------
            # Clear gradients
            # ------------------------------------------------
            optimizer.zero_grad()

            # ------------------------------------------------
            # Prediction
            # ------------------------------------------------
            predictions = ( model(xb).squeeze() )

            # ------------------------------------------------
            # Loss
            # ------------------------------------------------
            loss = criterion( predictions, yb.squeeze() )

            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------
            loss.backward()

            # ------------------------------------------------
            # Adam update
            # ------------------------------------------------
            optimizer.step()

        # 
        # Validation for Early Stopping
        # 
        model.eval()

        with torch.no_grad():
            val_preds_scaled = ( model( X_val_tensor ).squeeze() )

            val_loss = criterion( val_preds_scaled, y_val_tensor.squeeze() ).item()

        epochs_completed = ( epoch + 1 )

        # 
        # Early Stopping
        #
        # IMPORTANT:
        # Matches LSTM behavior.
        #
        # No model checkpoint restoration.
        # 
        if ( val_loss < best_val_loss ):
            best_val_loss = ( val_loss )

            epochs_no_improve = 0

        else:
            epochs_no_improve += 1

            if ( epochs_no_improve >= actual_patience ):
                break

    # --------------------------------------------------------
    # Ensure CUDA training finished
    # --------------------------------------------------------
    synchronize_device()

    training_time = ( time.perf_counter() - training_start )

    
    # Final Validation Prediction
    
    model.eval()

    with torch.no_grad():
        val_preds_scaled = ( model( X_val_tensor ).cpu().numpy() )

    # --------------------------------------------------------
    # Inverse Scale Prediction
    # --------------------------------------------------------
    val_preds_unscaled = ( scaler_y.inverse_transform( val_preds_scaled.reshape( -1, 1 ) ) )

    # --------------------------------------------------------
    # Inverse Scale Ground Truth
    # --------------------------------------------------------
    y_val_true_unscaled = ( scaler_y.inverse_transform( val_data[1].reshape( -1, 1 ) ) )              

    
    # Predictive Metrics
    

    r2 = EM.r2( y_val_true_unscaled, val_preds_unscaled )

    # --------------------------------------------------------
    # Handle invalid models
    # --------------------------------------------------------
    if ( np.isnan(r2) or np.isinf(r2) ):
        rmse = np.nan
        mae = np.nan
        acc = np.nan

    else:
        rmse = EM.rmse( y_val_true_unscaled, val_preds_unscaled )

        mae = EM.mae( y_val_true_unscaled, val_preds_unscaled )

        acc = EM.accuracy( y_val_true_unscaled, val_preds_unscaled )

    # --------------------------------------------------------
    # Profitability Index
    # --------------------------------------------------------
    profit_index,cop,cos = ( EM.profitability_index_v1( y_val_true_unscaled, val_preds_unscaled, forecast_window ) )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------
    del X_val_tensor
    del y_val_tensor

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------
    return (
        rmse,
        mae,
        r2,
        acc,
        profit_index,
        training_time,
        epochs_completed
    )



# Optuna Logging


optuna.logging.get_logger( "optuna" ).addHandler(
    logging.StreamHandler( sys.stdout )
)

optuna.logging.set_verbosity( optuna.logging.WARNING )



# Optuna Objective


def objective(
    trial,
    data,
    selected_indicators,
    ticker,
    window_size,
    forecast_window
):

    
    # Whole Trial Timer
    #
    # Same concept as LSTM optimization time.
    
    optimization_start = ( time.perf_counter() )

    
    # CPU Memory Before
    
    process = psutil.Process( os.getpid() )

    cpu_memory_before = ( process.memory_info().rss)

    
    # Reset GPU Peak Memory
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats( device )

    
    # Retrieve Data
    

    X_train = ( data["X_train"] )

    X_val = ( data["X_val"] )

    y_train_scaled = ( data["y_train"] )

    y_val_scaled = ( data["y_val"] )

    scaler_y = ( data["scaler_y"] )

    # --------------------------------------------------------
    # Test set optional
    # --------------------------------------------------------
    X_test = data.get( "X_test", None )

    hyperparams = {
        "lr": trial.suggest_categorical( "lr", [ 0.0001, 0.0005, 0.001 ] ),
        "batch_size": trial.suggest_categorical( "batch_size", [ 16, 32, 64 ] ),
        "epochs": trial.suggest_categorical( "epochs", [ 50, 100, 150 ] )
    }

    # DataLoader

    train_loader = DataLoader( TensorDataset(
            torch.tensor( X_train, dtype=torch.float32 ),
            torch.tensor( y_train_scaled, dtype=torch.float32 )
        ),
        batch_size=( hyperparams[ "batch_size" ] ),
        shuffle=False
    )
   
    input_size = ( X_train.shape[1] * X_train.shape[2] )

    
    # Create Model
    

    model = LinearRegressionModel( input_size=( input_size ) ).to(device)

    # Adam Optimizer

    optimizer = optim.Adam(
        model.parameters(),
        lr=( hyperparams[ "lr" ] )
    )

    # Loss

    criterion = nn.MSELoss()

    # Parameter Counts

    ( total_parameters, trainable_parameters ) = get_parameter_counts( model )

    # MACs / Sample

    macs_per_sample = ( estimate_linear_macs_per_sample( model ) )

    # Approx FLOPs / Sample
    
    approx_flops_per_sample = ( 2 * macs_per_sample )

    # Train

    rmse, mae, r2, acc, profit_index, training_time, epochs_completed = train_and_evaluate_model(
        model=model,
        train_loader=train_loader,
        val_data=( X_val, y_val_scaled ),
        optimizer=optimizer,
        criterion=criterion,
        scaler_y=scaler_y,
        forecast_window=( forecast_window ),
        epochs=( hyperparams[ "epochs" ] )
    )

    # Inference Benchmarks

    inference_batch_size = 256

    # --------------------------------------------------------
    # Training Inference
    # --------------------------------------------------------

    train_inference_time, train_latency_per_sample = measure_inference_time(
        model,
        X_train,
        batch_size=( inference_batch_size )
    )

    # Validation Inference

    validation_inference_time, validation_latency_per_sample = measure_inference_time(
        model,
        X_val,
        batch_size=( inference_batch_size )
    )

    # Test Inference

    if ( X_test is not None and len(X_test) > 0 ):       
        test_inference_time, test_latency_per_sample = measure_inference_time(
            model,
            X_test,
            batch_size=( inference_batch_size )
        )

    else:
        test_inference_time = ( np.nan )
        test_latency_per_sample = ( np.nan )
           
    # Peak GPU Memory


    if torch.cuda.is_available():
        synchronize_device()

        peak_gpu_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    else:
        peak_gpu_memory = ( np.nan )

    # CPU Memory After

    cpu_memory_after = ( process.memory_info().rss )

    cpu_memory_change = ( cpu_memory_after - cpu_memory_before ) / (1024 ** 2)

    
    # Complete Optuna Trial Time

    optimization_time = ( time.perf_counter() - optimization_start )

    
    # Store Optuna Attributes


    trial.set_user_attr( "training_time", training_time )

    trial.set_user_attr( "optimization_time", optimization_time )
    trial.set_user_attr( "total_parameters", total_parameters )
    trial.set_user_attr( "trainable_parameters", trainable_parameters )
    trial.set_user_attr( "macs_per_sample", macs_per_sample )
    trial.set_user_attr( "flops_per_sample", approx_flops_per_sample )
    trial.set_user_attr( "peak_gpu_memory_mb", peak_gpu_memory )
    trial.set_user_attr( "cpu_memory_change_mb", cpu_memory_change )

    # Write Results

    with open( 
        f'stock_results/'
        f'{ticker}_LR_results.csv',
        'a',
        newline=''
    ) as f:
        writer = csv.writer( f )

        writer.writerow([

            # Trial / Configuration

            trial.number,
            ', '.join( selected_indicators ),
            hyperparams[ "lr" ],
            hyperparams[ "batch_size" ],
            window_size,
            forecast_window,
            hyperparams[ "epochs" ],
            epochs_completed,

            # Predictive Performance

            rmse,
            mae,
            r2,
            acc,
            profit_index,

           
            # Timing


            round( training_time, 6 ),
            round( optimization_time, 6 ),
            round( train_inference_time, 7 ),
            round( validation_inference_time, 7 ),
            ( round( test_inference_time, 7 )
                if not np.isnan( test_inference_time )
                else np.nan
            ),
            test_latency_per_sample,

          
            # Complexity

            total_parameters,
            trainable_parameters,
            macs_per_sample,
            approx_flops_per_sample,

            # Memory

            peak_gpu_memory,
            cpu_memory_change
        ])

    
    # Cleanup  

    del model
    del optimizer
    del train_loader

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Optuna Objective    

    if ( np.isnan(rmse) or np.isinf(rmse) ):
        return 1e10
    return rmse



# Main Execution

if __name__ == "__main__":
    
    # Ticker
    #
    # Same ticker as current LSTM script.    

    ticker = 'AAPL'
    
    # Results Directory    

    os.makedirs( 'stock_results', exist_ok=True )
    
    # Result File    

    result_file = (
        f'stock_results/'
        f'{ticker}_LR_results.csv'
    )
    
    # Create CSV Header
    
    if not os.path.exists( result_file ):
        with open(
            result_file,
            'w',
            newline=''

        ) as f:
            writer = csv.writer( f )

            writer.writerow([

                # --------------------------------------------
                # Trial / Configuration
                # --------------------------------------------

                "Trial",
                "Indicators",
                "LR",
                "Batch Size",
                "Window Size",
                "Forecast Window",
                "Requested Epochs",
                "Actual Epochs",

                # --------------------------------------------
                # Predictive Metrics
                # --------------------------------------------

                "RMSE",
                "MAE",
                "R2",
                "Accuracy",
                "Profit Index",

                # --------------------------------------------
                # Time Metrics
                # --------------------------------------------

                "Training Time (s)",
                "Optimization Time (s)",
                "Train Inference Time (s)",
                "Validation Inference Time (s)",
                "Test Inference Time (s)",
                "Test Latency / Sample (s)",

                # --------------------------------------------
                # Complexity
                # --------------------------------------------

                "Total Parameters",
                "Trainable Parameters",
                "MACs / Sample",
                "Approx FLOPs / Sample",

                # --------------------------------------------
                # Memory
                # --------------------------------------------

                "Peak GPU Memory (MB)",
                "CPU Memory Change (MB)"
            ])

    
    # Preprocessing    

    data_processor = ( DataPreprocessing( ticker=ticker ) )

    df_with_all_indicators = ( data_processor.add_technical_indicators() )
    
    # Save Downloaded / Processed Data    

    os.makedirs( 'check', exist_ok=True )

    df_with_all_indicators.to_csv(
        f'check/'
        f'{ticker}_LR_downloaded_data.csv',
        index=True
    )
    
    # Indicators
    
    selected_base_indicators = [
        '20MA',
        '50MA',
        'RSI',
        'MACD',
        'Upper_BB',
        'Lower_BB',
        'CCI',
        'ATR',
        'Williams_%R',
        'OBV'
    ]

    
    # Select 6 Indicators from 10
    #
    # C(10,6) = 210 combinations    

    all_combinations = list( combinations( selected_base_indicators, 6 ) )

    window_forecast_combos = [ (60, 1), (60, 30), (5, 1) ]
    
    # Each configuration = 25 Optuna trials.
    
    start_combo_index = 135
    start_config_index = 0
    
    # Indicator Combination Loop    

    for i, indicator_combo in enumerate( all_combinations ):

        # ----------------------------------------------------
        # Resume
        # ----------------------------------------------------

        if i < start_combo_index:
            continue

        # 
        # Window / Forecast Loop
        # 

        for j, ( window_size, forecast_window ) in enumerate(
            window_forecast_combos
        ):

            # ------------------------------------------------
            # Resume
            # ------------------------------------------------

            if ( i == start_combo_index and j < start_config_index ):
                continue

            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            print( f"[j:{j}|cur:{i + 1}]" )

            # =
            # Prepare Data
            # =

            data = ( data_processor .prepare_data( ['Close'] + list( indicator_combo ), window_size=( window_size ), forecast_window=( forecast_window ) ) )

            # =
            # Check Dataset Shapes
            # =

            print( "X_train shape:", data[ "X_train" ].shape )
            print( "X_val shape:", data[ "X_val" ].shape )

            if ( "X_test" in data and data["X_test"] is not None ):
                print( "X_test shape:", data[ "X_test" ].shape )

            else:
                print(  "X_test not available." )

            # =
            # Optuna
            # =

            study = ( optuna.create_study( direction='minimize' ) )

            study.optimize(
                partial(
                    objective,
                    data=data,
                    selected_indicators=( indicator_combo ),
                    ticker=ticker,
                    window_size=( window_size ),
                    forecast_window=( forecast_window )
                ),

                # --------------------------------------------
                # Same as LSTM
                # --------------------------------------------

                n_trials=25
            )

            # =
            # Best Trial for Current Configuration
            # =

            print(
                "\nBest trial for this "
                "configuration:"
            )

            print(
                "Best RMSE:",
                study.best_value
            )

            print(
                "Best Parameters:",
                study.best_params
            )

    
    # Record Overall Best Models    

    record_best_models( result_file )

GPU Name: NVIDIA GeForce RTX 5080


[*********************100%***********************]  1 of 1 completed


Removed 61 rows corresponding to ex-dividend and earnings dates.
Null values in each column:
 Price        Ticker
Close        AAPL      0
High         AAPL      0
Low          AAPL      0
Open         AAPL      0
Volume       AAPL      0
20MA                   0
50MA                   0
RSI                    0
12EMA                  0
26EMA                  0
MACD                   0
Signal_Line            0
Upper_BB               0
Lower_BB               0
CCI                    0
ATR                    0
Williams_%R            0
OBV                    0
dtype: int64
Does the dataset contain any null values? False
[j:0|cur:136]
X_train shape: (1852, 60, 7)
X_val shape: (264, 60, 7)
X_test not available.

Best trial for this configuration:
Best RMSE: 5.52
Best Parameters: {'lr': 0.0001, 'batch_size': 32, 'epochs': 150}
[j:1|cur:136]
X_train shape: (1831, 60, 7)
X_val shape: (261, 60, 7)
X_test not available.

Best trial for this configuration:
Best RMSE: 16.769
Best Parameters: {'lr'